# Predictive AI Evaluation Challenge — First Submission

**Pipeline:**
1. Download `aims-foundations/measurement-db` from HuggingFace
2. Explore & preprocess into training format
3. Train NCF model (sentence-transformer embeddings + MLP)
4. Evaluate on validation set
5. Create submission ZIP for Codabench

## 1. Imports

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from datasets import Features, Value, load_dataset
from huggingface_hub import HfApi
import torch
import torch.nn as nn
from sentence_transformers import SentenceTransformer
from sklearn.model_selection import train_test_split
from sklearn.metrics import log_loss, roc_auc_score
import warnings
warnings.filterwarnings('ignore')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
print(f'PyTorch: {torch.__version__}')

/home/azureuser/gpt4hana-base/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cuda
PyTorch: 2.10.0+cu128


## 2. Download Dataset from HuggingFace

In [2]:
REPO_ID = "aims-foundations/measurement-db"
REGISTRY_FILES = {"subjects.parquet", "items.parquet", "benchmarks.parquet"}

# List all parquet files in the repo
repo_files = HfApi().list_repo_files(repo_id=REPO_ID, repo_type="dataset")

# Filter to response tables only (not registry, not traces)
response_files = sorted(
    name for name in repo_files
    if name.endswith(".parquet")
    and name not in REGISTRY_FILES
    and not name.endswith("_traces.parquet")
)

print(f"Found {len(response_files)} response files:")
for f in response_files:
    print(f"  {f}")

Found 16 response files:
  afrimedqa.parquet
  agentdojo.parquet
  ai2d_test.parquet
  androidworld.parquet
  bfcl.parquet
  cybench.parquet
  hle.parquet
  livecodebench.parquet
  matharena.parquet
  mathvista_mini.parquet
  mmbench_v11.parquet
  mmlupro.parquet
  mtbench.parquet
  rewardbench.parquet
  swebench.parquet
  ultrafeedback.parquet


In [3]:
# Load registry tables
items = load_dataset(REPO_ID, data_files="items.parquet", split="train")
subjects = load_dataset(REPO_ID, data_files="subjects.parquet", split="train")
benchmarks = load_dataset(REPO_ID, data_files="benchmarks.parquet", split="train")

print(f"Items: {len(items)} rows, columns: {items.column_names}")
print(f"Subjects: {len(subjects)} rows, columns: {subjects.column_names}")
print(f"Benchmarks: {len(benchmarks)} rows, columns: {benchmarks.column_names}")

Repo card metadata block was not found. Setting CardData to empty.
Generating train split: 103983 examples [00:00, 419598.67 examples/s]
Repo card metadata block was not found. Setting CardData to empty.
Generating train split: 909 examples [00:00, 162696.18 examples/s]
Repo card metadata block was not found. Setting CardData to empty.
Generating train split: 16 examples [00:00, 1928.08 examples/s]

Items: 103983 rows, columns: ['item_id', 'benchmark_id', 'raw_item_id', 'content', 'correct_answer', 'content_hash']
Subjects: 909 rows, columns: ['subject_id', 'display_name', 'provider', 'hub_repo', 'revision', 'params', 'release_date', 'raw_labels_seen', 'notes']
Benchmarks: 16 rows, columns: ['benchmark_id', 'name', 'version', 'license', 'source_url', 'description', 'modality', 'domain', 'response_type', 'response_scale', 'categorical', 'paper_url', 'release_date']


In [4]:
# Load response tables with explicit schema
response_features = Features({
    "subject_id": Value("string"),
    "item_id": Value("string"),
    "benchmark_id": Value("string"),
    "trial": Value("int64"),
    "test_condition": Value("string"),
    "response": Value("float64"),
    "correct_answer": Value("string"),
    "trace": Value("string"),
})

responses = load_dataset(
    REPO_ID,
    data_files=response_files,
    features=response_features,
    split="train",
)
print(f"Responses: {len(responses)} rows")
print(f"Columns: {responses.column_names}")

Repo card metadata block was not found. Setting CardData to empty.
Generating train split: 5361379 examples [00:02, 2408714.31 examples/s]

Responses: 5361379 rows
Columns: ['subject_id', 'item_id', 'benchmark_id', 'trial', 'test_condition', 'response', 'correct_answer', 'trace']


## 3. Explore & Build Training Data

In [5]:
# Build lookup dicts
items_by_id = {row["item_id"]: row for row in items}
subjects_by_id = {row["subject_id"]: row for row in subjects}
benchmarks_by_id = {row["benchmark_id"]: row for row in benchmarks}

print(f"Unique items: {len(items_by_id)}")
print(f"Unique subjects: {len(subjects_by_id)}")
print(f"Unique benchmarks: {len(benchmarks_by_id)}")

# Show sample subject
sample_subj = list(subjects_by_id.values())[0]
print(f"\nSample subject keys: {list(sample_subj.keys())}")
print(f"Sample subject: {sample_subj}")

Unique items: 103983
Unique subjects: 909
Unique benchmarks: 16

Sample subject keys: ['subject_id', 'display_name', 'provider', 'hub_repo', 'revision', 'params', 'release_date', 'raw_labels_seen', 'notes']
Sample subject: {'subject_id': '05788e96b1593db5', 'display_name': 'BioMistral-7B', 'provider': None, 'hub_repo': None, 'revision': None, 'params': None, 'release_date': None, 'raw_labels_seen': ['BioMistral-7B'], 'notes': None}


In [6]:
# Render subject_content the same way the platform does
def render_subject_content(subject, fallback_subject_id):
    display_name = subject.get("display_name") or fallback_subject_id
    lines = [f"Name: {display_name}"]
    optional_fields = (
        ("provider", "Organization"),
        ("params", "Parameters"),
        ("release_date", "Released"),
        ("family", "Family"),
    )
    for key, label in optional_fields:
        value = subject.get(key)
        if value:
            lines.append(f"{label}: {value}")
    return "\n".join(lines)

# Convert a response row to the training format matching predict() input
def to_training_example(row):
    item = items_by_id.get(row["item_id"], {})
    subject = subjects_by_id.get(row["subject_id"], {})
    benchmark = benchmarks_by_id.get(row["benchmark_id"], {})
    benchmark_id = benchmark.get("benchmark_id") or row["benchmark_id"]
    
    item_content = item.get("content")
    if item_content is None:
        return None  # skip rows with no item text
    
    resp = row["response"]
    if resp is None or np.isnan(resp):
        return None
    
    return {
        "benchmark": benchmark_id,
        "condition": row["test_condition"] or "none",
        "subject_content": render_subject_content(subject, row["subject_id"]),
        "item_content": item_content,
        "label": int(resp),
    }

# Test on one row
sample = to_training_example(responses[0])
if sample:
    for k, v in sample.items():
        print(f"{k}: {str(v)[:120]}")

benchmark: afrimedqa
condition: source=afrimedqa-v2|prompt=base
subject_content: Name: BioMistral-7B
item_content: You are counseling the family of a 15-year-old female with a BMI of 38 and type II diabetes on the surgical options for 
label: 1


In [7]:
# Convert all responses to training examples
# This may take a few minutes for large datasets
print("Converting responses to training format...")
train_data = []
skipped = 0
for i, row in enumerate(responses):
    ex = to_training_example(row)
    if ex is not None:
        train_data.append(ex)
    else:
        skipped += 1
    if (i + 1) % 500_000 == 0:
        print(f"  Processed {i+1:,} rows ({len(train_data):,} kept, {skipped:,} skipped)")

df = pd.DataFrame(train_data)
print(f"\nFinal training set: {len(df):,} rows, {skipped:,} skipped")
print(f"Label distribution:\n{df['label'].value_counts()}")
print(f"Unique benchmarks: {df['benchmark'].nunique()}")
print(f"Unique subjects: {df['subject_content'].nunique()}")
df.head()

Converting responses to training format...
  Processed 500,000 rows (500,000 kept, 0 skipped)
  Processed 1,000,000 rows (1,000,000 kept, 0 skipped)
  Processed 1,500,000 rows (1,500,000 kept, 0 skipped)
  Processed 2,000,000 rows (2,000,000 kept, 0 skipped)
  Processed 2,500,000 rows (2,500,000 kept, 0 skipped)
  Processed 3,000,000 rows (3,000,000 kept, 0 skipped)
  Processed 3,500,000 rows (3,500,000 kept, 0 skipped)
  Processed 4,000,000 rows (4,000,000 kept, 0 skipped)
  Processed 4,500,000 rows (4,500,000 kept, 0 skipped)
  Processed 5,000,000 rows (5,000,000 kept, 0 skipped)

Final training set: 5,361,379 rows, 0 skipped
Label distribution:
label
1     2901180
0     1547332
5      402559
4      232255
3      168634
2      106339
10       1137
9         808
8         522
7         331
6         282
Name: count, dtype: int64
Unique benchmarks: 16
Unique subjects: 909


,benchmark,condition,subject_content,item_content,label
0,afrimedqa,source=afrimedqa-v2|prompt=base,Name: BioMistral-7B,You are counseling the family of a 15-year-old...,1
1,afrimedqa,source=afrimedqa-v2|prompt=base,Name: BioMistral-7B,A 14-year-old male is being assessed for weigh...,0
2,afrimedqa,source=afrimedqa-v2|prompt=base,Name: BioMistral-7B,\r\n A 15-year-old female presents to a specia...,0
3,afrimedqa,source=afrimedqa-v2|prompt=base,Name: BioMistral-7B,Following resection of a patent urachus and bl...,0
4,afrimedqa,source=afrimedqa-v2|prompt=base,Name: BioMistral-7B,A 1-week old term baby presents with clear flu...,1


In [8]:
# Binarize labels: some benchmarks have non-binary scores (e.g., 0-5 scale)
# The competition uses binary {0, 1}, so treat anything > 0 as 1, or filter to binary only
print(f"Label value counts before binarize:\n{df['label'].value_counts().head(10)}")

# Option: keep only binary responses (safest for first submission)
df_binary = df[df['label'].isin([0, 1])].copy()
print(f"\nAfter filtering to binary only: {len(df_binary):,} rows (dropped {len(df) - len(df_binary):,})")
print(f"Label distribution:\n{df_binary['label'].value_counts()}")

# Use binary subset
df = df_binary.reset_index(drop=True)
df['label'] = df['label'].astype(int)

Label value counts before binarize:
label
1     2901180
0     1547332
5      402559
4      232255
3      168634
2      106339
10       1137
9         808
8         522
7         331
Name: count, dtype: int64

After filtering to binary only: 4,448,512 rows (dropped 912,867)
Label distribution:
label
1    2901180
0    1547332
Name: count, dtype: int64


## 4. Build Per-Subject Mean Baseline + NCF Model

We'll build two approaches:
1. **Per-subject mean accuracy** — simple but strong baseline
2. **NCF** — sentence-transformer embeddings + MLP head

In [9]:
# ---- Per-subject mean accuracy (our fallback) ----
subject_means = df.groupby('subject_content')['label'].mean().to_dict()
global_mean = df['label'].mean()
print(f"Global mean accuracy: {global_mean:.4f}")
print(f"Unique subjects with means: {len(subject_means)}")

# Quick validation: split by items (cold-start simulation)
unique_items = df['item_content'].unique()
train_items, val_items = train_test_split(unique_items, test_size=0.1, random_state=42)
train_items_set = set(train_items)
val_items_set = set(val_items)

df_train = df[df['item_content'].isin(train_items_set)].copy()
df_val = df[df['item_content'].isin(val_items_set)].copy()
print(f"\nTrain: {len(df_train):,} rows | Val: {len(df_val):,} rows")

# Per-subject mean baseline on validation
subject_means_train = df_train.groupby('subject_content')['label'].mean().to_dict()
global_mean_train = df_train['label'].mean()

df_val['pred_mean'] = df_val['subject_content'].map(subject_means_train).fillna(global_mean_train)
df_val['pred_mean'] = df_val['pred_mean'].clip(0.01, 0.99)

baseline_logloss = log_loss(df_val['label'], df_val['pred_mean'])
baseline_auc = roc_auc_score(df_val['label'], df_val['pred_mean'])
print(f"\nPer-subject mean baseline:")
print(f"  Log-loss: {baseline_logloss:.4f} (neg: {-baseline_logloss:.4f})")
print(f"  AUC-ROC:  {baseline_auc:.4f}")

Global mean accuracy: 0.6522
Unique subjects with means: 909

Train: 4,009,641 rows | Val: 438,871 rows

Per-subject mean baseline:
  Log-loss: 0.5776 (neg: -0.5776)
  AUC-ROC:  0.7095


In [10]:
# ---- Compute sentence embeddings ----
# We embed unique subjects and items only once
print("Loading sentence transformer...")
encoder = SentenceTransformer("all-mpnet-base-v2", device=DEVICE)

# Get unique texts
unique_subjects_list = df_train['subject_content'].unique().tolist()
unique_items_list = df_train['item_content'].unique().tolist()

# Also include val subjects/items for embedding lookup
val_subjects_extra = [s for s in df_val['subject_content'].unique() if s not in set(unique_subjects_list)]
val_items_extra = [s for s in df_val['item_content'].unique() if s not in set(unique_items_list)]

all_subjects_list = unique_subjects_list + val_subjects_extra
all_items_list = unique_items_list + val_items_extra

print(f"Encoding {len(all_subjects_list)} unique subjects...")
subject_embeddings = encoder.encode(all_subjects_list, batch_size=64, show_progress_bar=True, convert_to_numpy=True)

print(f"Encoding {len(all_items_list)} unique items...")
item_embeddings = encoder.encode(all_items_list, batch_size=256, show_progress_bar=True, convert_to_numpy=True)

# Build lookup dicts
subj_emb_dict = {text: emb for text, emb in zip(all_subjects_list, subject_embeddings)}
item_emb_dict = {text: emb for text, emb in zip(all_items_list, item_embeddings)}

EMB_DIM = subject_embeddings.shape[1]
print(f"Embedding dim: {EMB_DIM}")

Loading sentence transformer...
Encoding 909 unique subjects...


Batches: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 15/15 [00:00<00:00, 21.51it/s]


Encoding 70834 unique items...


Batches: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 277/277 [04:49<00:00,  1.04s/it]


Embedding dim: 768


In [11]:
# ---- Build training tensors ----
def build_tensors(dataframe, subj_dict, item_dict):
    U, V, Y = [], [], []
    skipped = 0
    for _, row in dataframe.iterrows():
        se = subj_dict.get(row['subject_content'])
        ie = item_dict.get(row['item_content'])
        if se is None or ie is None:
            skipped += 1
            continue
        U.append(se)
        V.append(ie)
        Y.append(row['label'])
    print(f"  Built {len(Y)} samples, skipped {skipped}")
    return (
        torch.tensor(np.array(U), dtype=torch.float32),
        torch.tensor(np.array(V), dtype=torch.float32),
        torch.tensor(np.array(Y), dtype=torch.float32),
    )

# If dataset is very large, sample for training
MAX_TRAIN = 500_000
if len(df_train) > MAX_TRAIN:
    print(f"Sampling {MAX_TRAIN:,} from {len(df_train):,} training rows")
    df_train_sample = df_train.sample(MAX_TRAIN, random_state=42)
else:
    df_train_sample = df_train

MAX_VAL = 50_000
if len(df_val) > MAX_VAL:
    df_val_sample = df_val.sample(MAX_VAL, random_state=42)
else:
    df_val_sample = df_val

print("Building training tensors...")
U_train, V_train, Y_train = build_tensors(df_train_sample, subj_emb_dict, item_emb_dict)
print("Building validation tensors...")
U_val, V_val, Y_val = build_tensors(df_val_sample, subj_emb_dict, item_emb_dict)

Sampling 500,000 from 4,009,641 training rows
Building training tensors...
  Built 500000 samples, skipped 0
Building validation tensors...
  Built 50000 samples, skipped 0


In [12]:
# ---- Define and train NCF MLP ----
class NCFHead(nn.Module):
    def __init__(self, emb_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2 * emb_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 1),
        )
    
    def forward(self, u, v):
        x = torch.cat([u, v], dim=-1)
        return self.net(x).squeeze(-1)

ncf = NCFHead(EMB_DIM).to(DEVICE)
optimizer = torch.optim.Adam(ncf.parameters(), lr=1e-3, weight_decay=1e-5)
criterion = nn.BCEWithLogitsLoss()

# Training loop
BATCH_SIZE = 2048
EPOCHS = 10
n_train = len(Y_train)

for epoch in range(EPOCHS):
    ncf.train()
    perm = torch.randperm(n_train)
    epoch_loss = 0.0
    n_batches = 0
    
    for i in range(0, n_train, BATCH_SIZE):
        idx = perm[i:i+BATCH_SIZE]
        u_b = U_train[idx].to(DEVICE)
        v_b = V_train[idx].to(DEVICE)
        y_b = Y_train[idx].to(DEVICE)
        
        logits = ncf(u_b, v_b)
        loss = criterion(logits, y_b)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
        n_batches += 1
    
    # Validation
    ncf.eval()
    with torch.no_grad():
        val_logits = ncf(U_val.to(DEVICE), V_val.to(DEVICE))
        val_probs = torch.sigmoid(val_logits).cpu().numpy()
        val_probs = np.clip(val_probs, 1e-6, 1 - 1e-6)
        val_ll = log_loss(Y_val.numpy(), val_probs)
        val_auc = roc_auc_score(Y_val.numpy(), val_probs)
    
    print(f"Epoch {epoch+1}/{EPOCHS} | Train loss: {epoch_loss/n_batches:.4f} | "
          f"Val log-loss: {val_ll:.4f} (neg: {-val_ll:.4f}) | Val AUC: {val_auc:.4f}")

print(f"\n--- Comparison ---")
print(f"Per-subject mean baseline: log-loss={baseline_logloss:.4f}, AUC={baseline_auc:.4f}")
print(f"NCF model:                 log-loss={val_ll:.4f}, AUC={val_auc:.4f}")

Epoch 1/10 | Train loss: 0.5482 | Val log-loss: 0.5091 (neg: -0.5091) | Val AUC: 0.7997
Epoch 2/10 | Train loss: 0.4838 | Val log-loss: 0.4867 (neg: -0.4867) | Val AUC: 0.8183
Epoch 3/10 | Train loss: 0.4591 | Val log-loss: 0.4770 (neg: -0.4770) | Val AUC: 0.8272
Epoch 4/10 | Train loss: 0.4414 | Val log-loss: 0.4733 (neg: -0.4733) | Val AUC: 0.8315
Epoch 5/10 | Train loss: 0.4280 | Val log-loss: 0.4697 (neg: -0.4697) | Val AUC: 0.8334
Epoch 6/10 | Train loss: 0.4166 | Val log-loss: 0.4703 (neg: -0.4703) | Val AUC: 0.8344
Epoch 7/10 | Train loss: 0.4075 | Val log-loss: 0.4740 (neg: -0.4740) | Val AUC: 0.8345
Epoch 8/10 | Train loss: 0.4006 | Val log-loss: 0.4741 (neg: -0.4741) | Val AUC: 0.8339
Epoch 9/10 | Train loss: 0.3941 | Val log-loss: 0.4781 (neg: -0.4781) | Val AUC: 0.8340
Epoch 10/10 | Train loss: 0.3879 | Val log-loss: 0.4859 (neg: -0.4859) | Val AUC: 0.8322

--- Comparison ---
Per-subject mean baseline: log-loss=0.5776, AUC=0.7095
NCF model:                 log-loss=0.4859, 

## 5. Save Model & Build Submission ZIP

In [13]:
# Save the trained NCF head and subject mean lookup
import pickle, json

SUBMISSION_DIR = Path("/home/azureuser/measurement_science_v2/competition/my_submission")
SUBMISSION_DIR.mkdir(exist_ok=True)

# Save NCF weights
torch.save(ncf.cpu().state_dict(), SUBMISSION_DIR / "ncf_head.pt")
print(f"Saved NCF weights: {(SUBMISSION_DIR / 'ncf_head.pt').stat().st_size / 1024:.0f} KB")

# Save per-subject mean accuracy as fallback
# Retrain on full data
subject_means_full = df.groupby('subject_content')['label'].mean().to_dict()
global_mean_full = df['label'].mean()

with open(SUBMISSION_DIR / "subject_means.pkl", "wb") as f:
    pickle.dump({"means": subject_means_full, "global": global_mean_full}, f)
print(f"Saved subject means: {(SUBMISSION_DIR / 'subject_means.pkl').stat().st_size / 1024:.0f} KB")

Saved NCF weights: 1669 KB
Saved subject means: 35 KB


In [14]:
# Write model.py for the submission
model_py = '''\
"""NCF submission for Predictive AI Evaluation Challenge."""
from __future__ import annotations
import pickle
import numpy as np
import torch
import torch.nn as nn
from pathlib import Path
from sentence_transformers import SentenceTransformer

# ---------- Module-level init (runs once) ----------
_DIR = Path(__file__).parent

# Load sentence transformer
ENCODER = SentenceTransformer("all-mpnet-base-v2")
EMB_DIM = 768

# Load trained NCF head
class NCFHead(nn.Module):
    def __init__(self, emb_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2 * emb_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 1),
        )
    def forward(self, u, v):
        x = torch.cat([u, v], dim=-1)
        return self.net(x).squeeze(-1)

NCF = NCFHead(EMB_DIM)
NCF.load_state_dict(torch.load(_DIR / "ncf_head.pt", map_location="cpu", weights_only=True))
NCF.eval()

# Load per-subject mean fallback
with open(_DIR / "subject_means.pkl", "rb") as f:
    _sm = pickle.load(f)
SUBJECT_MEANS = _sm["means"]
GLOBAL_MEAN = _sm["global"]

# Cache embeddings to avoid re-encoding the same text
_emb_cache = {}

def _encode(text: str) -> np.ndarray:
    if text not in _emb_cache:
        _emb_cache[text] = ENCODER.encode(text, convert_to_numpy=True)
    return _emb_cache[text]

# ---------- Required entry point ----------
def predict(input: dict, labeled: list[dict] | None = None) -> float:
    """Return predicted P(subject answers item correctly)."""
    try:
        u = _encode(input["subject_content"])
        v = _encode(input["item_content"])
        u_t = torch.tensor(u, dtype=torch.float32).unsqueeze(0)
        v_t = torch.tensor(v, dtype=torch.float32).unsqueeze(0)
        with torch.no_grad():
            logit = NCF(u_t, v_t)
            prob = torch.sigmoid(logit).item()
        # Clip to avoid log(0)
        return float(max(0.01, min(0.99, prob)))
    except Exception:
        # Fallback to per-subject mean
        mean = SUBJECT_MEANS.get(input.get("subject_content", ""), GLOBAL_MEAN)
        return float(max(0.01, min(0.99, mean)))
'''

with open(SUBMISSION_DIR / "model.py", "w") as f:
    f.write(model_py)
print("Wrote model.py")

Wrote model.py


In [15]:
# Write labeling.py (simple random — we can improve later)
labeling_py = '''\
"""Adaptive labeling: random baseline (no special acquisition strategy)."""
from __future__ import annotations
import random

def acquisition_function(input: dict) -> float:
    return random.random()
'''

with open(SUBMISSION_DIR / "labeling.py", "w") as f:
    f.write(labeling_py)
print("Wrote labeling.py")

Wrote labeling.py


In [16]:
# Create the submission ZIP
import zipfile

ZIP_PATH = Path("/home/azureuser/measurement_science_v2/competition/submission_v1.zip")

with zipfile.ZipFile(ZIP_PATH, 'w', zipfile.ZIP_DEFLATED) as zf:
    for fpath in sorted(SUBMISSION_DIR.iterdir()):
        zf.write(fpath, fpath.name)
        print(f"  Added: {fpath.name} ({fpath.stat().st_size / 1024:.1f} KB)")

print(f"\n✅ Submission ZIP created: {ZIP_PATH}")
print(f"   Size: {ZIP_PATH.stat().st_size / 1024:.1f} KB")

  Added: labeling.py (0.2 KB)
  Added: model.py (2.2 KB)
  Added: ncf_head.pt (1669.0 KB)
  Added: subject_means.pkl (34.5 KB)

✅ Submission ZIP created: /home/azureuser/measurement_science_v2/competition/submission_v1.zip
   Size: 1521.0 KB


In [17]:
# Verify ZIP contents
with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
    print("ZIP contents:")
    for info in zf.infolist():
        print(f"  {info.filename:30s} {info.file_size/1024:8.1f} KB")

print(f"\n🎯 Ready to upload: {ZIP_PATH}")
print("Go to: https://aimslab.stanford.edu/competition/submit")
print("Upload submission_v1.zip and check the leaderboard!")

ZIP contents:
  labeling.py                         0.2 KB
  model.py                            2.2 KB
  ncf_head.pt                      1669.0 KB
  subject_means.pkl                  34.5 KB

🎯 Ready to upload: /home/azureuser/measurement_science_v2/competition/submission_v1.zip
Go to: https://aimslab.stanford.edu/competition/submit
Upload submission_v1.zip and check the leaderboard!
